# Snippet from Cookbook.md


In [ ]:
#!/usr/bin/env python3
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from typing import Optional, Dict, Any
from compitum import Router, Config
import logging
app = FastAPI(title="Compitum Routing API", version="1.0.0")
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
config = Config.from_yaml('configs/production.yaml')
router = Router(config)
class RouteRequest(BaseModel):
    prompt: str = Field(..., description="Query to route")
    context: Optional[str] = Field(None, description="Additional context")
    max_steps: Optional[int] = Field(5, ge=1, le=20)
    trace: bool = Field(False, description="Include certificate")
class RouteResponse(BaseModel):
    response: str
    model: str
    utility: float
    certificate: Optional[Dict[str, Any]] = None
@app.post("/route", response_model=RouteResponse)
async def route_prompt(request: RouteRequest):
    try:
        result = router.route(
            prompt=request.prompt,
            context=request.context,
            max_steps=request.max_steps
        )
        response_data = {
            "response": result.response,
            "model": result.certificate['selected_model'],
            "utility": result.certificate['utility']
        }
        if request.trace:
            response_data["certificate"] = result.certificate
        return RouteResponse(**response_data)
    except Exception as e:
        logger.error(f"Routing error: {e}")
        raise HTTPException(status_code=500, detail=str(e))
@app.get("/health")
async def health_check():
    return {"status": "healthy", "router": "ready"}
@app.get("/config")
async def get_config():
    return config.to_dict()
@app.post("/config/update")
async def update_config(constraint_name: str, value: float):
    try:
        config.constraints[constraint_name] = value
        router.reload_config(config)
        return {"message": f"Updated {constraint_name} = {value}"}
    except KeyError:
        raise HTTPException(status_code=404, detail=f"Constraint '{constraint_name}' not found")
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
